# Laya vs Phi-4-mini on BoolQ

Standalone Colab notebook. It needs no repo checkout.

**Question:** on short-passage yes/no reading comprehension, how does a 421M-param non-autoregressive decision model compare with the best small LLM from the [GSM8K SLM experiment](https://github.com/CJosh88/slms/tree/main/experiments/2026-09-slm-gsm8k), on accuracy, calibration, speed and memory?

| Model | Params | Type | Hugging Face id | Licence |
|---|---|---|---|---|
| Laya | 421M | ModernBERT-large decision model, one forward pass, 512-token window | [`convaiinnovations/laya`](https://huggingface.co/convaiinnovations/laya) | Apache-2.0 |
| Phi-4-mini-instruct | 3.8B | Autoregressive chat LLM (88.5% on GSM8K in the SLM experiment) | [`microsoft/Phi-4-mini-instruct`](https://huggingface.co/microsoft/Phi-4-mini-instruct) | MIT |

**Dataset:** [BoolQ](https://huggingface.co/datasets/google/boolq) (`google/boolq`, `validation` split, CC-BY-SA-3.0). It has naturally occurring yes/no questions, each paired with a Wikipedia passage (~110 tokens on average). The notebook uses 500 randomly chosen examples (seed 42).

**Method**
- Laya gets the passage as its `state` and the question as one `noul` (yes/no) question. It returns P(yes).
- Phi-4-mini gets the same passage and question in a chat prompt, and decoding is greedy. P(yes) comes from the first answer token's logits for *Yes* vs *No*, so both models produce a probability and can be compared on AUC and calibration, not just accuracy.
- Both models see the **same passage text**. The few passages longer than Laya's window are truncated to `PASSAGE_TOKEN_BUDGET` tokens *for both models*, and the results report them separately.

**Runtime → Change runtime type → T4 GPU** before running. Model outputs are cached in `cache/`, so a rerun after a disconnect resumes where it stopped.

In [ ]:
%pip install -q "laya>=0.3.20" "transformers==4.57.3" "accelerate==1.11.0" "datasets==4.8.5" scikit-learn scipy

In [ ]:
import gc, json, math, random, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch, transformers, laya
from datasets import load_dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    # T4 (compute capability 7.5) has no native bf16, so use fp16 there.
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
else:
    DTYPE = torch.float32
print("device:", DEVICE, "| dtype:", DTYPE)

## 1. Config

In [ ]:
N_SAMPLES = 500               # 3,270 validation examples in total; None = all
THRESHOLD = 0.5               # Yes when P(yes) >= THRESHOLD (both models)
PASSAGE_TOKEN_BUDGET = 400    # passages longer than this (Laya tokens) are truncated for both models

LAYA_ID = "convaiinnovations/laya"
PHI_ID = "microsoft/Phi-4-mini-instruct"
PHI_BATCH = 8                 # lower this if you hit CUDA out-of-memory errors

SYSTEM_PROMPT = "You answer yes/no questions about a passage."
USER_TEMPLATE = "Passage:\n{passage}\n\nQuestion: {question}\nAnswer with only Yes or No."

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/slms/2026-09-laya-vs-phi4-boolq")
else:
    BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "cache"
FIG_DIR = BASE_DIR / "figures"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load BoolQ

In [ ]:
raw = load_dataset("google/boolq", split="validation")
print(raw)

n = len(raw) if N_SAMPLES is None else N_SAMPLES
sample_idx = sorted(np.random.default_rng(SEED).choice(len(raw), size=n, replace=False).tolist())


def normalise_question(q: str) -> str:
    # BoolQ questions are lowercase with no question mark.
    q = q.strip()
    return q[0].upper() + q[1:] + ("" if q.endswith("?") else "?")


df = pd.DataFrame({
    "idx": sample_idx,
    "question": [normalise_question(raw[i]["question"]) for i in sample_idx],
    "passage": [raw[i]["passage"] for i in sample_idx],
    "label": [bool(raw[i]["answer"]) for i in sample_idx],
})
MAJORITY = df.label.mean()
print(f"{len(df)} examples | Yes rate {MAJORITY:.1%} (majority-class baseline accuracy = {max(MAJORITY, 1 - MAJORITY):.1%})")
for _, r in df.head(2).iterrows():
    print(f"\n[{r.idx}] {r.question} -> {r.label}\n{r.passage[:300]}...")

## 3. Load Laya, smoke test, and check token lengths

Laya's English checkpoint has a 512-token window shared by the question, the options and the state. This step measures each passage with Laya's own tokenizer and truncates the rare long ones to `PASSAGE_TOKEN_BUDGET`.

In [ ]:
agent = laya.load(LAYA_ID, device=DEVICE)

res = agent.predict(
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, completed in 1889.",
    {"q": {"type": "noul", "instructions": "Is the Eiffel Tower in Paris?"},
     "q2": {"type": "noul", "instructions": "Was the Eiffel Tower completed in the 20th century?"}},
)
for k, v in res["answers"].items():
    print(k, "P(yes) =", round(v["noul"], 3))

In [ ]:
from transformers import AutoTokenizer

laya_tok = getattr(agent, "tokenizer", None)
if laya_tok is None:  # fall back to the backbone's tokenizer
    laya_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-large")


def n_tokens(text):
    return len(laya_tok(text, add_special_tokens=False)["input_ids"])


def truncate(text, budget):
    ids = laya_tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= budget else laya_tok.decode(ids[:budget])


df["passage_tokens"] = df.passage.map(n_tokens)
df["question_tokens"] = df.question.map(n_tokens)
df["truncated"] = df.passage_tokens > PASSAGE_TOKEN_BUDGET
df["passage_in"] = [truncate(p, PASSAGE_TOKEN_BUDGET) for p in df.passage]

print(df[["passage_tokens", "question_tokens"]].describe().round(1).T)
print(f"\nTruncated to {PASSAGE_TOKEN_BUDGET} tokens: {df.truncated.sum()} / {len(df)} ({df.truncated.mean():.1%})")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(df.passage_tokens + df.question_tokens, bins=40, color="#4C78A8")
ax.axvline(512, color="crimson", ls="--", label="Laya window (512)")
ax.set_xlabel("Passage + question tokens (Laya tokenizer)"); ax.set_ylabel("Examples"); ax.legend()
ax.set_title("BoolQ input lengths")
fig.savefig(FIG_DIR / "token_lengths.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Run Laya

Each example has its own question, so examples are scored one at a time (batch size 1). This is Laya's realistic single-request latency. If a long input still overflows the window, the passage is re-truncated to 300 tokens and retried.

In [ ]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["idx"]: rec for rec in map(json.loads, f)}


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


laya_path = CACHE_DIR / "laya.jsonl"
done = load_cache(laya_path)
todo = df[~df.idx.isin(done)]
print(f"Laya: {len(done)} cached, {len(todo)} to run")

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
with laya_path.open("a", encoding="utf-8") as f:
    for _, r in tqdm(todo.iterrows(), total=len(todo), desc="Laya"):
        q = {"q": {"type": "noul", "instructions": r.question}}
        err = None
        sync(); t0 = time.perf_counter()
        try:
            p_yes = agent.predict(r.passage_in, q)["answers"]["q"]["noul"]
        except (RuntimeError, ValueError) as e:
            err = str(e)
            p_yes = agent.predict(truncate(r.passage_in, 300), q)["answers"]["q"]["noul"]
        sync(); latency = time.perf_counter() - t0
        f.write(json.dumps({"idx": int(r.idx), "p_yes": float(p_yes), "latency_s": latency, "retry_error": err}) + "\n")

laya_meta = {"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else float("nan"),
             "params_m": sum(p.numel() for p in agent.model.parameters()) / 1e6 if hasattr(agent, "model") else 421.0}
if todo.empty and (CACHE_DIR / "laya_meta.json").exists():
    laya_meta = json.loads((CACHE_DIR / "laya_meta.json").read_text())
(CACHE_DIR / "laya_meta.json").write_text(json.dumps(laya_meta))
print(laya_meta)

del agent
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 5. Run Phi-4-mini

Decoding is greedy with `max_new_tokens=3`. P(yes) is the softmax over the first generated position's logits, restricted to the *Yes* and *No* tokens. The decoded text is still parsed, and the parse rate shows how often the model followed the format. When the text doesn't start with yes/no, the prediction falls back to P(yes) ≥ threshold.

In [ ]:
from transformers import AutoModelForCausalLM

phi_path = CACHE_DIR / "phi4.jsonl"
done = load_cache(phi_path)
todo = df[~df.idx.isin(done)]
print(f"Phi-4-mini: {len(done)} cached, {len(todo)} to run")


def first_token_ids(tok, words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})


def parse_answer(text):
    t = text.strip().lower().lstrip("*'\"").strip()
    return True if t.startswith("yes") else False if t.startswith("no") else None


phi_meta_path = CACHE_DIR / "phi4_meta.json"
if len(todo):
    tok = AutoTokenizer.from_pretrained(PHI_ID)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(PHI_ID, dtype=DTYPE, device_map="auto").eval()
    YES_IDS = first_token_ids(tok, ["Yes", " Yes", "yes", " yes", "YES"])
    NO_IDS = first_token_ids(tok, ["No", " No", "no", " no", "NO"])
    print("yes ids", YES_IDS, "| no ids", NO_IDS)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    with phi_path.open("a", encoding="utf-8") as f:
        for start in tqdm(range(0, len(todo), PHI_BATCH), desc="Phi-4-mini"):
            batch = todo.iloc[start:start + PHI_BATCH]
            prompts = [
                tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": USER_TEMPLATE.format(passage=p, question=q)}],
                    tokenize=False, add_generation_prompt=True)
                for p, q in zip(batch.passage_in, batch.question)
            ]
            enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
            sync(); t0 = time.perf_counter()
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=3, do_sample=False, output_scores=True,
                                     return_dict_in_generate=True, pad_token_id=tok.pad_token_id)
            sync(); per = (time.perf_counter() - t0) / len(batch)
            logits = out.scores[0].float()
            lse_yes = torch.logsumexp(logits[:, YES_IDS], dim=-1)
            lse_no = torch.logsumexp(logits[:, NO_IDS], dim=-1)
            p_yes = torch.sigmoid(lse_yes - lse_no).tolist()
            texts = tok.batch_decode(out.sequences[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for idx, t, p, n_in in zip(batch.idx, texts, p_yes, enc["attention_mask"].sum(1).tolist()):
                f.write(json.dumps({"idx": int(idx), "output": t, "p_yes": p, "latency_s": per,
                                    "input_tokens": int(n_in)}) + "\n")

    phi_meta = {"peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else float("nan"),
                "params_m": sum(p.numel() for p in model.parameters()) / 1e6}
    phi_meta_path.write_text(json.dumps(phi_meta))
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
phi_meta = json.loads(phi_meta_path.read_text())
print(phi_meta)

## 6. Results

In [ ]:
from sklearn.metrics import brier_score_loss, f1_score, roc_auc_score


def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


def ece(y, p, bins=10):
    # Expected calibration error of P(yes) against the Yes label.
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, bins + 1)
    which = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    return sum(abs(y[which == b].mean() - p[which == b].mean()) * (which == b).mean()
               for b in range(bins) if (which == b).any())


laya_df = pd.DataFrame(load_cache(laya_path).values())
laya_df["pred"] = laya_df.p_yes >= THRESHOLD
laya_df["model"] = "Laya"

phi_df = pd.DataFrame(load_cache(phi_path).values())
phi_df["parsed"] = phi_df.output.map(parse_answer)
phi_df["pred"] = [a if a is not None else p >= THRESHOLD for a, p in zip(phi_df.parsed, phi_df.p_yes)]
phi_df["model"] = "Phi-4-mini"

res = pd.concat([laya_df, phi_df], ignore_index=True).merge(
    df[["idx", "question", "label", "truncated", "passage_tokens"]], on="idx")
res = res[res.idx.isin(df.idx)]
res["correct"] = res.pred == res.label
res.to_csv(BASE_DIR / "results.csv", index=False)

metas = {"Laya": laya_meta, "Phi-4-mini": phi_meta}
rows = [{"model": "Majority (always Yes)" if MAJORITY >= 0.5 else "Majority (always No)", "n": len(df),
         "accuracy": max(MAJORITY, 1 - MAJORITY)}]
for name, g in res.groupby("model", sort=False):
    lo, hi = wilson(g.correct.sum(), len(g))
    rows.append({
        "model": name, "n": len(g), "accuracy": g.correct.mean(), "ci_low": lo, "ci_high": hi,
        "f1_yes": f1_score(g.label, g.pred), "roc_auc": roc_auc_score(g.label, g.p_yes),
        "brier": brier_score_loss(g.label, g.p_yes), "ece": ece(g.label, g.p_yes),
        "acc_truncated": g[g.truncated].correct.mean() if g.truncated.any() else float("nan"),
        "ms_per_example": 1000 * g.latency_s.mean(), "params_m": metas[name]["params_m"],
        "peak_mem_gb": metas[name]["peak_mem_gb"],
    })
summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(BASE_DIR / "summary.csv")
print(f"Phi-4-mini answered in Yes/No format: {phi_df.parsed.notna().mean():.1%}")
print(f"Laya: batch size 1 | Phi-4-mini: batch size {PHI_BATCH} (latency is per example, amortised over the batch)")
summary.style.format({
    "accuracy": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "f1_yes": "{:.3f}", "roc_auc": "{:.3f}",
    "brier": "{:.3f}", "ece": "{:.3f}", "acc_truncated": "{:.1%}", "ms_per_example": "{:.1f}",
    "params_m": "{:,.0f}", "peak_mem_gb": "{:.2f}",
}, na_rep="–")

### Paired comparison

Both models answered the same questions, so the exact McNemar test on the examples where exactly one model was right shows whether the accuracy gap is real.

In [ ]:
from scipy.stats import binomtest

wide = res.pivot(index="idx", columns="model", values="correct")
only_laya = int((wide["Laya"] & ~wide["Phi-4-mini"]).sum())
only_phi = int((~wide["Laya"] & wide["Phi-4-mini"]).sum())
print("Both right:     ", int((wide["Laya"] & wide["Phi-4-mini"]).sum()))
print("Only Laya right:", only_laya)
print("Only Phi right: ", only_phi)
print("Both wrong:     ", int((~wide["Laya"] & ~wide["Phi-4-mini"]).sum()))
if only_laya + only_phi:
    p = binomtest(only_laya, only_laya + only_phi, 0.5).pvalue
    print(f"\nExact McNemar p = {p:.4f}", "(significant at 0.05)" if p < 0.05 else "(not significant at 0.05)")

### Threshold sweep

Both models output P(yes), so the 0.5 threshold is only a default. The sweep shows how much accuracy each model gains from tuning it. The tuned numbers are optimistic because the threshold is chosen on the test sample itself.

In [ ]:
ths = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = pd.DataFrame([
    {"threshold": t, "model": name, "accuracy": ((g.p_yes >= t) == g.label).mean(),
     "f1_yes": f1_score(g.label, g.p_yes >= t, zero_division=0)}
    for name, g in res.groupby("model", sort=False) for t in ths
])
best = sweep.loc[sweep.groupby("model").accuracy.idxmax()]
print(best.to_string(index=False))
sweep.pivot(index="threshold", columns="model", values="accuracy").style.format("{:.1%}").highlight_max(axis=0)

## 7. Figures

In [ ]:
COLORS = {"Laya": "#4C78A8", "Phi-4-mini": "#F58518"}
models = [m for m in summary.index if m in COLORS]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
s = summary.loc[models]
ax.bar(models, s.accuracy, yerr=[s.accuracy - s.ci_low, s.ci_high - s.accuracy], capsize=6,
       color=[COLORS[m] for m in models])
ax.axhline(max(MAJORITY, 1 - MAJORITY), color="grey", ls="--", label="majority baseline")
ax.set_ylim(0, 1); ax.set_ylabel("Accuracy"); ax.legend(loc="lower right")
ax.set_title(f"BoolQ accuracy (n={len(df)}, 95% Wilson CI)")
for i, v in enumerate(s.accuracy):
    ax.text(i, v + 0.03, f"{v:.1%}", ha="center")

ax = axes[1]
for m in models:
    ax.scatter(s.loc[m, "ms_per_example"], s.loc[m, "accuracy"], s=max(s.loc[m, "peak_mem_gb"], 0.5) * 80,
               color=COLORS[m], alpha=0.7)
    ax.annotate(f"{m}\n{s.loc[m, 'params_m']:,.0f}M params, {s.loc[m, 'peak_mem_gb']:.1f} GB",
                (s.loc[m, "ms_per_example"], s.loc[m, "accuracy"]), textcoords="offset points", xytext=(10, -4))
ax.set_xscale("log"); ax.set_xlabel("ms per example (log scale)"); ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs latency (bubble = peak GPU memory)")

ax = axes[2]
edges = np.linspace(0, 1, 11)
for m in models:
    g = res[res.model == m]
    b = np.clip(np.digitize(g.p_yes, edges[1:-1]), 0, 9)
    cal = pd.DataFrame({"b": b, "p": g.p_yes, "y": g.label.astype(float)}).groupby("b").mean()
    ax.plot(cal.p, cal.y, "o-", color=COLORS[m], label=f"{m} (ECE {summary.loc[m, 'ece']:.3f})")
ax.plot([0, 1], [0, 1], color="grey", ls=":")
ax.set_xlabel("Predicted P(yes)"); ax.set_ylabel("Observed Yes rate"); ax.legend()
ax.set_title("Reliability diagram")

plt.tight_layout()
fig.savefig(FIG_DIR / "accuracy_speed_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

### Disagreements

In [ ]:
dis = wide.index[wide["Laya"] != wide["Phi-4-mini"]][:6]
by = res.set_index(["idx", "model"])
for idx in dis:
    q = df.set_index("idx").loc[idx]
    print("=" * 100)
    print(f"[{idx}] {q.question}  GOLD: {'Yes' if q.label else 'No'}\n{q.passage[:400]}...")
    for m in ["Laya", "Phi-4-mini"]:
        r = by.loc[(idx, m)]
        print(f"  {'OK ' if r.correct else 'BAD'} {m:10s} P(yes)={r.p_yes:.3f}")

## 8. Save results

Download the zip and commit `figures/`, `results.csv` and `summary.csv` to this experiment folder. Set `USE_DRIVE = True` in the config to write everything straight to Drive.

In [ ]:
import shutil
bundle = Path("/tmp/laya_vs_phi4_boolq_results")
if bundle.exists():
    shutil.rmtree(bundle)
for name in ["cache", "figures"]:
    shutil.copytree(BASE_DIR / name, bundle / name)
for name in ["results.csv", "summary.csv"]:
    shutil.copy(BASE_DIR / name, bundle / name)
shutil.make_archive("laya_vs_phi4_boolq_results", "zip", bundle)
try:
    from google.colab import files
    files.download("laya_vs_phi4_boolq_results.zip")
except ImportError:
    print("Not on Colab; results in", BASE_DIR.resolve())